# Projet Kayak — Partie 1 : Récupération de la météo

**Objectif :** pour une liste de 35 villes touristiques françaises, récupérer :
1. Les coordonnées géographiques (latitude / longitude) via l'API **Nominatim** (OpenStreetMap) — gratuite, sans clé.
2. Les prévisions météo sur 5 jours via l'API **OpenWeatherMap** — gratuite, nécessite une clé API.

On en déduira ensuite un **score météo** par ville pour identifier les destinations les plus ensoleillées de la semaine à venir.

## 1. Imports et configuration

In [1]:
%pip install python-dotenv plotly scrapy boto3 sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

# On charge la clé API OpenWeatherMap depuis un fichier .env (jamais commit sur git)
# Le fichier .env doit contenir une ligne : OWM_API_KEY=ta_cle_ici
load_dotenv()
OWM_API_KEY = os.getenv("OWM_API_KEY")

assert OWM_API_KEY, "Clé API OpenWeatherMap manquante — vérifie ton fichier .env"

## 2. Liste des 35 villes cibles

Liste fournie par l'énoncé du projet.

In [3]:
cities = [
    "Mont Saint Michel", "Saint Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy",
    "Grenoble", "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis",
    "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
    "Aigues Mortes", "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"
]

print(f"Nombre de villes : {len(cities)}")

Nombre de villes : 35


## 3. Géocoding avec Nominatim

Nominatim est l'API de géocodage d'OpenStreetMap. Elle est **gratuite et sans clé**, mais impose deux règles :
- Un `User-Agent` personnalisé (sinon la requête est refusée).
- **Maximum 1 requête par seconde** (politique d'utilisation).

On définit une fonction qui prend le nom d'une ville et renvoie un dict `{city, lat, lon}`.

In [4]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
HEADERS = {"User-Agent": "KayakProject/1.0 (projet etudiant)"}

def geocode(city: str) -> dict:
    """Retourne lat/lon pour une ville française via Nominatim."""
    params = {
        "q": f"{city}, France",  # on précise France pour éviter les homonymes
        "format": "json",
        "limit": 1,                # on ne garde que le premier résultat
    }
    response = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=10)
    response.raise_for_status()
    results = response.json()
    print(type(results))   # list -> Nominatim renvoie un tableau JSON de résultats 
    print(results)         # renvoie tout le résultat (ici toutes les infos concernant Paris quand on appelle la fonction avec Paris)
    if not results:        # fonctionne exactement comme un if / else, grâce à un mécanisme qu'on appelle le early return (retour anticipé)
        return {"city": city, "lat": None, "lon": None}
    return {
        "city": city,      # renvoie un dictionnaire avec simplement ce dont on a besoin, i.e. ville, lat, long
        "lat": float(results[0]["lat"]),
        "lon": float(results[0]["lon"]),
    }

# Test sur une ville
geocode("Paris")

<class 'list'>
[{'place_id': 98427244, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'relation', 'osm_id': 7444, 'lat': '48.8588897', 'lon': '2.3200410', 'class': 'boundary', 'type': 'administrative', 'place_rank': 15, 'importance': 0.897098092136026, 'addresstype': 'suburb', 'name': 'Paris', 'display_name': 'Paris, Île-de-France, France métropolitaine, France', 'boundingbox': ['48.8155755', '48.9021560', '2.2241220', '2.4697602']}]


{'city': 'Paris', 'lat': 48.8588897, 'lon': 2.320041}

In [5]:
# On boucle sur toutes les villes en respectant le rate limit (1 req/sec)
coords = []
for city in cities:     # cities est la liste des 35 villes référencées plus haut 
    coords.append(geocode(city))
    time.sleep(1)       # indispensable pour ne pas se faire bannir

df_coords = pd.DataFrame(coords)
df_coords

<class 'list'>
[{'place_id': 281361808, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'way', 'osm_id': 211285890, 'lat': '48.6359541', 'lon': '-1.5114600', 'class': 'place', 'type': 'islet', 'place_rank': 20, 'importance': 0.4711675122514022, 'addresstype': 'islet', 'name': 'Mont Saint-Michel', 'display_name': 'Mont Saint-Michel, Le Mont-Saint-Michel, Avranches, Manche, Normandie, France métropolitaine, 50170, France', 'boundingbox': ['48.6349172', '48.6370310', '-1.5133292', '-1.5094796']}]
<class 'list'>
[{'place_id': 284162046, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'relation', 'osm_id': 905534, 'lat': '48.6495180', 'lon': '-2.0260409', 'class': 'boundary', 'type': 'administrative', 'place_rank': 16, 'importance': 0.6250994319975981, 'addresstype': 'town', 'name': 'Saint-Malo', 'display_name': 'Saint-Malo, Ille-et-Vilaine, Bretagne, France métropolitaine, 35400, France', 'bound

,city,lat,lon
0,Mont Saint Michel,48.635954,-1.511460
1,Saint Malo,48.649518,-2.026041
2,Bayeux,49.276462,-0.702474
3,Le Havre,49.493898,0.107973
4,Rouen,49.440459,1.093966
5,Paris,48.858890,2.320041
6,Amiens,49.894171,2.295695
7,Lille,50.636565,3.063528
8,Strasbourg,48.584614,7.750713
9,Chateau du Haut Koenigsbourg,48.249382,7.343941


In [6]:
# Vérification : aucune ville sans coordonnées ?
missing = df_coords[df_coords["lat"].isna()]  # on utilise un mask -> un Df conditionné à l'intérieur d'un Df 
print(f"Villes sans coordonnées : {len(missing)}")
if len(missing) > 0:
    print(missing)

Villes sans coordonnées : 0


## 4. Prévisions météo via OpenWeatherMap

On utilise l'endpoint **`/data/2.5/forecast`** qui renvoie les prévisions sur **5 jours avec un pas de 3 heures** (donc 40 points par ville). C'est l'endpoint gratuit le plus riche — il ne nécessite pas de carte bancaire.

Paramètres clés :
- `lat`, `lon` : coordonnées de la ville
- `appid` : notre clé API
- `units=metric` : températures en °C
- `lang=fr` : descriptions météo en français

In [7]:
# On définit une fonction qui prend un couple de coordonnées d'une ville et renvoie une liste de 40 points de prévision sous forme de dict
OWM_URL = "https://api.openweathermap.org/data/2.5/forecast"

def get_forecast(lat: float, lon: float) -> list:
    """Retourne la liste brute des prévisions 3h sur 5 jours pour un point."""
    params = {
        "lat": lat,
        "lon": lon,
        "appid": OWM_API_KEY,
        "units": "metric",
        "lang": "fr",
    }
    response = requests.get(OWM_URL, params=params, timeout=10)
    response.raise_for_status()
    return response.json()["list"]  # liste de 40 prévisions hebdo (8 x 5j) pour une ville donnée 

# Test sur Paris
sample = get_forecast(48.8566, 2.3522)  # à quoi ressemble le format des prévisions pour UNE seule ville 
print(f"Nombre de points de prévision : {len(sample)}")
sample[0]  # on regarde UNE seule prévision parmi les 40 (8 x 5j)
            # et on ne met pas de print() à la dernière ligne d'une cellule de notebook jupiter, pas besoin, elle est automatiquement affichée

Nombre de points de prévision : 40


{'dt': 1782669600,
 'main': {'temp': 29.85,
  'feels_like': 29.99,
  'temp_min': 27.53,
  'temp_max': 29.85,
  'pressure': 1019,
  'sea_level': 1019,
  'grnd_level': 1010,
  'humidity': 44,
  'temp_kf': 2.32,
  'dew_point': 16.29},
 'weather': [{'id': 500,
   'main': 'Rain',
   'description': 'légère pluie',
   'icon': '10d'}],
 'clouds': {'all': 77},
 'wind': {'speed': 4.16, 'deg': 5, 'gust': 5.22},
 'visibility': 10000,
 'pop': 0.2,
 'rain': {'3h': 0.15},
 'sys': {'pod': 'd'},
 'dt_txt': '2026-06-28 18:00:00'}

### Agrégation des prévisions par ville

Chaque appel renvoie 40 points (5 jours × 8 points/jour). Pour scorer une ville, on résume ces 40 points en quelques indicateurs :
- **temp_avg** : température moyenne sur 5 jours (°C)
- **rain_total** : pluie cumulée sur 5 jours (mm)
- **clouds_avg** : couverture nuageuse moyenne (%)

In [8]:
# on définit une fonction qui prend une liste de 40 prévisions hebdo et retourne un dictionnaire d'indicateurs synthétiques
def summarize_forecast(forecast_list: list) -> dict:
    """Agrège une liste de 40 prévisions en quelques indicateurs synthétiques."""
    temps = [p["main"]["temp"] for p in forecast_list]  # list comprehension -> on prend la température (x40) 
    clouds = [p["clouds"]["all"] for p in forecast_list]   # on prend la couverture nuageuse (x40)
    # La pluie n'est présente dans le JSON que s'il pleut — d'où le .get()
    rain = [p.get("rain", {}).get("3h", 0) for p in forecast_list]   # on prend la pluie (x40)
    return {
        "temp_avg": round(sum(temps) / len(temps), 1),
        "rain_total": round(sum(rain), 1),
        "clouds_avg": round(sum(clouds) / len(clouds), 1),
    }

summarize_forecast(sample)

{'temp_avg': 23.3, 'rain_total': 1.0, 'clouds_avg': 37.6}

In [9]:
# On boucle sur toutes les villes qui ont des coordonnées
weather_data = []
for _, row in df_coords.iterrows():
    if pd.isna(row["lat"]):
        continue
    forecast = get_forecast(row["lat"], row["lon"])
    summary = summarize_forecast(forecast)
    weather_data.append({
        "city": row["city"],
        "lat": row["lat"],
        "lon": row["lon"],
        **summary,
    })
    # OpenWeatherMap autorise 60 req/min en gratuit : on est large, mais on reste propre
    time.sleep(0.2)

df_weather = pd.DataFrame(weather_data)
df_weather

,city,lat,lon,temp_avg,rain_total,clouds_avg
0,Mont Saint Michel,48.635954,-1.511460,17.3,0.0,47.0
1,Saint Malo,48.649518,-2.026041,17.1,0.0,54.0
2,Bayeux,49.276462,-0.702474,18.0,0.0,42.9
3,Le Havre,49.493898,0.107973,17.8,0.0,41.3
4,Rouen,49.440459,1.093966,19.7,0.0,42.6
5,Paris,48.858890,2.320041,23.3,1.0,37.6
6,Amiens,49.894171,2.295695,18.7,0.0,42.2
7,Lille,50.636565,3.063528,19.2,0.2,43.5
8,Strasbourg,48.584614,7.750713,24.0,11.7,49.4
9,Chateau du Haut Koenigsbourg,48.249382,7.343941,20.7,16.1,41.5


In [10]:
sample = get_forecast(48.8566, 2.3522)  # Paris
print(f"Première prévision : {sample[0]['dt_txt']}")
print(f"Dernière prévision : {sample[-1]['dt_txt']}")
print(f"Nombre de points    : {len(sample)}")

Première prévision : 2026-06-28 18:00:00
Dernière prévision : 2026-07-03 15:00:00
Nombre de points    : 40


## 5. Calcul d'un score météo

On construit un score simple qui récompense la chaleur et pénalise la pluie et les nuages.
Formule choisie (arbitraire mais explicable) :

$$\text{score} = \text{temp\_avg} - 0.5 \times \text{rain\_total} - 0.1 \times \text{clouds\_avg}$$

Plus le score est élevé, plus la destination est attractive météo.

In [11]:
df_weather["weather_score"] = (
    df_weather["temp_avg"]
    - 0.5 * df_weather["rain_total"]
    - 0.1 * df_weather["clouds_avg"]
).round(2)

# Top 5 des destinations de la semaine
df_weather.sort_values("weather_score", ascending=False).head()

,city,lat,lon,temp_avg,rain_total,clouds_avg,weather_score
20,Marseille,43.296399,5.377789,28.8,0.0,11.2,27.68
21,Aix en Provence,43.529842,5.447474,29.7,0.1,20.6,27.59
22,Avignon,43.949249,4.805901,29.8,1.4,15.4,27.56
24,Nimes,43.837425,4.360069,30.0,1.8,15.6,27.54
25,Aigues Mortes,43.566152,4.191540,29.0,0.2,17.2,27.18


## 6. Sauvegarde

On sauvegarde le DataFrame en CSV pour éviter de relancer tous les appels API si on rouvre le notebook plus tard.

In [12]:
df_weather.to_csv("data/weather.csv", index=False)
print("✅ Données sauvegardées dans data/weather.csv")

✅ Données sauvegardées dans data/weather.csv
